## 1. 라이브러리 및 환경 설정

In [2]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from lightgbm import LGBMRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error

In [3]:
%pip install xgboost

Note: you may need to restart the kernel to use updated packages.


In [4]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import StackingRegressor
# 스태킹에 사용할 base 모델들 정의
base_models = [
    ('lgb', LGBMRegressor(n_estimators=1000, learning_rate=0.05, max_depth=7,
                          num_leaves=63, subsample=0.8, colsample_bytree=0.8,
                          reg_alpha=0.1, reg_lambda=0.1, random_state=42, verbose=-1)),
    ('rf', RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)),
    ('xgb', XGBRegressor(n_estimators=1000, learning_rate=0.05, max_depth=7, 
                         subsample=0.8, colsample_bytree=0.8, random_state=42, verbosity=0))
]

# 메타 모델 (스태킹 레벨 2)
meta_model = Ridge(alpha=1.0)
print(f"베이스 모델 개수: {len(base_models)}")
print(f"메타 모델: {meta_model.__class__.__name__}")

베이스 모델 개수: 3
메타 모델: Ridge


## 2. 데이터 로드

In [5]:
from sklearn.preprocessing import RobustScaler
train = pd.read_csv('./data/train.csv')
test = pd.read_csv('./data/test.csv')

print(f"학습 데이터 크기: {train.shape}")
print(f"테스트 데이터 크기: {test.shape}")

학습 데이터 크기: (250000, 94)
테스트 데이터 크기: (50000, 93)


## 3. 피처 및 타겟 설정

In [6]:
TARGET = 'avg_delay_minutes_next_30m'
ID_COLS = ['ID', 'layout_id', 'scenario_id']

feature_cols = [c for c in train.columns if c not in ID_COLS + [TARGET]]
print(f"피처 수: {len(feature_cols)}")

rs = RobustScaler()
train[feature_cols] = rs.fit_transform(train[feature_cols])
test[feature_cols] = rs.transform(test[feature_cols])


피처 수: 90


## 4. 모델 학습 (5-Fold CV)

In [8]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

stacking_model = StackingRegressor(base_models,final_estimator=meta_model,cv=kf)
stacking_model.fit(train[feature_cols],train[TARGET])
predict = stacking_model.predict(train[feature_cols])
mean_absolute_error(train[TARGET],predict)

6.128364696172718

In [15]:
predict_test = stacking_model.predict(test[feature_cols])
predict_test

array([15.03616592, 15.06841027, 17.52702164, ..., 10.19125618,
       19.43122977, 12.15565879], shape=(50000,))

## 5. 결과 확인

## 6. 제출 파일 생성

In [16]:
submission = pd.DataFrame({'ID': test['ID'], TARGET: predict_test})
submission.to_csv('./submission.csv', index=False)
print("submission.csv 저장 완료.")

submission.csv 저장 완료.
